# 📘 NB_01_Load_CityData

> Ingest reference city data into the Lakehouse (`raw` schema) and log metadata.



#### 📒 Notebook Summary  
**Loading City Location Data & Logging Metadata in Microsoft Fabric**

📌 Step 1: Set Parameters

- Defines Lakehouse name and ID.
- Specifies source URL for city data: `worldcities.csv` from GitHub.
- Declares schema and table names:
  - Raw Table: `raw.city_locations`
  - Log Table: `log.monitoring_log`

🧭 Step 2: Resolve Paths

- Constructs `target_path` and `log_path` using `abfss://` pattern.
- Uses Fabric `workspace_id` and `lakehouse_id`.
- Allows Delta I/O without requiring an attached Lakehouse.

 📥 Step 3: Load Source Data

- Downloads city location CSV from GitHub using Pandas.
- Converts the Pandas DataFrame to a Spark DataFrame.

 💾 Step 4: Write to Raw Layer

- Writes Spark DataFrame to Delta table at:
  - `abfss://<workspace_id>@onelake.dfs.fabric.microsoft.com/<lakehouse_id>/Tables/raw/city_locations`
- Uses:
  - `format("delta")`
  - `mode("overwrite")`
  - `mergeSchema = true`

 🧮 Step 5: Record Row Count

- Captures `row_count` after successful write.
- Tracks success/failure status and error message for log entry.

🪵 Step 6: Write Log Entry

- Creates a structured log entry with:
  - `notebook_name`, `table_name`, `run_id`, `user_name`, `operation_type`
  - `row_count`, `message`, `error_message`, `timestamp`
- Appends log to Delta table:
  - `abfss://<workspace_id>@onelake.dfs.fabric.microsoft.com/<lakehouse_id>/Tables/log/monitoring_log`


In [ ]:
import pandas as pd
import datetime
import uuid
import sempy.fabric as fabric
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

In [ ]:
# -----------------------------
# Parameters
# -----------------------------
lakehouse_id = fabric.get_lakehouse_id()
workspace_id = fabric.get_workspace_id()
lakehouse_name_resolved = fabric.resolve_item_name(lakehouse_id)
notebook_name = fabric.resolve_item_name(fabric.get_artifact_id())

In [ ]:
source_url = "https://raw.githubusercontent.com/condwanaland/worldcities/refs/heads/main/data-raw/worldcities.csv"

# Define schemas
raw_schema = "raw"
log_schema = "log"

# Table names
log_table_name = "monitoring_log"
table_name = "city_locations"

# Qualified Table names
qualified_table_name = f"{raw_schema}.{table_name}"
qualified_log_table_name = f"{log_schema}.{log_table_name}"

print(f"📦 Table: {table_name}")
print(f"📦 Log Table: {log_table_name}")

# -----------------------------
# Resolve ABFS Paths
# -----------------------------
workspace_id = fabric.get_notebook_workspace_id()

# ✅ Correct ABFS paths using schema folders
target_path = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables/{raw_schema}/{table_name}"
log_path = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables/{log_schema}/{log_table_name}"

print(f"📁 Target ABFS path: {target_path}")
print(f"📁 Log ABFS path:    {log_path}")

In [ ]:
# -----------------------------
# Load Data from URL
# -----------------------------
try:
    print(f"📥 Loading data from: {source_url}")
    df_pd = pd.read_csv(source_url)
    print(f"✅ Loaded {len(df_pd)} rows")

    df_spark = spark.createDataFrame(df_pd)
    print(f"🔄 Converted to Spark DataFrame")

    print(f"💾 Writing to ABFS path: {target_path}")
    df_spark.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(qualified_table_name)
    print("✅ Write successful")

    load_status = "success"
    error_message = ""
    row_count = df_spark.count()

except Exception as e:
    load_status = "failure"
    error_message = str(e)
    row_count = 0
    print(f"❌ Failed: {error_message}")

In [ ]:
# -----------------------------
# Logging
# -----------------------------
try:
    run_id = str(uuid.uuid4())
    timestamp = datetime.datetime.utcnow()
    notebook_name = fabric.resolve_item_name(fabric.get_artifact_id()) or "UnknownNotebook"

    log_schema_struct = StructType([
        StructField("notebook_name", StringType(), True),
        StructField("table_name", StringType(), True),
        StructField("run_id", StringType(), True),
        StructField("user_name", StringType(), True),
        StructField("operation_type", StringType(), True),
        StructField("row_count", IntegerType(), True),
        StructField("message", StringType(), True),
        StructField("error_message", StringType(), True),
        StructField("timestamp", TimestampType(), True)
    ])

    log_entry = [{
        "notebook_name": notebook_name,
        "table_name": f"{raw_schema}.{table_name}",
        "run_id": run_id,
        "user_name": "prathy.k",
        "operation_type": "load_world_cities",
        "row_count": row_count,
        "message": "Data load completed" if load_status == "success" else "Data load failed",
        "error_message": error_message,
        "timestamp": timestamp
    }]

    log_df = spark.createDataFrame(log_entry, schema=log_schema_struct)
    log_df.write.format("delta").mode("append").saveAsTable(qualified_log_table_name)
    print(f"🪵 Log written to: {log_path}")

except Exception as log_error:
    print(f"⚠️ Log write failed: {str(log_error)}")